# Backtesting Value-at-Risk (VaR)

VaR models must be validated to ensure they provide accurate risk estimates. One standard approach is the Kupiec Proportion of Failures (POF) test:

- Count how often actual losses exceed the VaR (breaches).
- Under a correct model, breaches should occur at a ~alpha fraction of the time (in this case 5%).
- Use a likelihood ratio test to check if the observed breach rate is consistent with expectations.

Included:
1. Compute 95% 1-day VaR using Historical, Parametric, and Monte Carlo methods.
2. Backtest against actual returns.
3. Apply Kupiec’s POF test.
4. Compare reliability between methods.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf


from src.var_methods import historical_var, parametric_var, monte_carlo_var
from src.backtesting import kupiec_test

## Load Data
We use a simple 4-asset portfolio as before: SPY, BND, GLD, QQQ

In [2]:
tickers = ["SPY", "BND", "GLD", "QQQ"]
data = pd.read_csv(
    "../data/sample_prices.csv",
    index_col=0,
    parse_dates=[0],
    date_parser=lambda x: pd.to_datetime(x, format="%d/%m/%Y")
)
log_returns = np.log(data / data.shift(1)).dropna()

weights = np.array([0.25, 0.25, 0.25, 0.25])
portfolio_returns = log_returns.dot(weights)
portfolio_returns.head()

Date
2020-01-03   -0.000014
2020-01-06    0.004870
2020-01-07    0.000035
2020-01-08    0.000932
2020-01-09    0.002679
dtype: float64

## Compute Daily VaR Estimates
We calculate 95% 1-day VaR using the three methods.

In [3]:
alpha = 0.05
portfolio_value = 1_000_000

hist_var = historical_var(log_returns, weights, alpha=alpha, horizon=1, portfolio_value=portfolio_value)
param_var = parametric_var(log_returns, weights, alpha=alpha, horizon=1, portfolio_value=portfolio_value)
mc_var = monte_carlo_var(log_returns, weights, alpha=alpha, horizon=1, sims=10000, portfolio_value=portfolio_value, seed=42)

print(f"Historical VaR (95%, 1-day): ${hist_var:,.2f}")
print(f"Parametric VaR (95%, 1-day): ${param_var:,.2f}")
print(f"Monte Carlo VaR (95%, 1-day): ${mc_var:,.2f}")

Historical VaR (95%, 1-day): $12,804.78
Parametric VaR (95%, 1-day): $13,448.11
Monte Carlo VaR (95%, 1-day): $13,504.69


## Count Breaches
We check how often actual returns exceed the VaR thresholds.

In [4]:
breaches_hist = (portfolio_returns < -hist_var/portfolio_value).sum()
breaches_param = (portfolio_returns < -param_var/portfolio_value).sum()
breaches_mc = (portfolio_returns < -mc_var/portfolio_value).sum()

total_obs = len(portfolio_returns)

print("Breaches:")
print(f"Historical: {breaches_hist} out of {total_obs}")
print(f"Parametric: {breaches_param} out of {total_obs}")
print(f"Monte Carlo: {breaches_mc} out of {total_obs}")

Breaches:
Historical: 63 out of 1257
Parametric: 56 out of 1257
Monte Carlo: 56 out of 1257


## Kupiec Test
We apply the POF test to evaluate whether the observed breach rate matches the expected 5%.

In [5]:
LR_hist, p_hist = kupiec_test(breaches_hist, total_obs, alpha)
LR_param, p_param = kupiec_test(breaches_param, total_obs, alpha)
LR_mc, p_mc = kupiec_test(breaches_mc, total_obs, alpha)

results = pd.DataFrame({
    "Method": ["Historical", "Parametric", "Monte Carlo"],
    "Breaches": [breaches_hist, breaches_param, breaches_mc],
    "p-value": [p_hist, p_param, p_mc]
})

results

,Method,Breaches,p-value
0,Historical,63,0.984518
1,Parametric,56,0.366792
2,Monte Carlo,56,0.366792


## Interpretation
- The p-value indicates whether the observed number of breaches is statistically consistent with the expected 5%.
- If p < 0.05, the model may be rejected (with too few/too many breaches).
- A good model should have breach rates close to 5% and not be rejected by Kupiec’s test.